# OrdonezB Binary Sensor Data ETL

Pipeline: raw text → parse & clean → join ADL labels → fact/dimension tables → save as CSV/Parquet

- Input: `OrdonezB_Sensors.txt` (sensor events), `OrdonezB_ADLs.txt` (activity-of-daily-living labels)
- Output: `events_fact`, `sensor_dim`, `activity_dim`, `adl_intervals` in the `processed/` folder

## 1. Environment Setup

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path(r"C:\Users\USER\Desktop\[16-08-26] Cloudy\07.Binary Dataset")
SENSOR_PATH = DATA_DIR / "OrdonezB_Sensors.txt"
ADL_PATH = DATA_DIR / "OrdonezB_ADLs.txt"
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"pandas {pd.__version__}")
print(f"output -> {OUTPUT_DIR}")

pandas 2.3.3
output -> C:\Users\USER\Desktop\[16-08-26] Cloudy\07.Binary Dataset\processed


## 2. Extract — Parse the Raw Text Files

- Skip the first two lines (header/separator)
- Column delimiters are irregular (1–2 tabs, mixed `\t \t`) → split on `\t+` and drop empty fields
- Rows with an unexpected number of columns are rejected

In [2]:
def parse_ordonez(path, columns):
    rows = []
    rejected = 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if i < 2 or not line.strip():
                continue
            parts = [p for p in (x.strip() for x in re.split(r"\t+", line.strip())) if p]
            if len(parts) == len(columns):
                rows.append(parts)
            else:
                rejected += 1
    df = pd.DataFrame(rows, columns=columns)
    if rejected:
        print(f"{path.name}: {rejected} row(s) rejected")
    return df

sensors_raw = parse_ordonez(SENSOR_PATH, ["start_time", "end_time", "location", "type", "place"])
adls_raw = parse_ordonez(ADL_PATH, ["start_time", "end_time", "activity"])

print(f"sensors_raw: {sensors_raw.shape}")
print(f"adls_raw: {adls_raw.shape}")
sensors_raw.head()

sensors_raw: (2334, 5)
adls_raw: (493, 3)


,start_time,end_time,location,type,place
0,2012-11-11 21:14:21,2012-11-12 00:21:49,Seat,Pressure,Living
1,2012-11-12 00:22:57,2012-11-12 00:22:59,Door,PIR,Living
2,2012-11-12 00:23:14,2012-11-12 00:23:17,Door,PIR,Kitchen
3,2012-11-12 00:24:20,2012-11-12 00:24:22,Door,PIR,Kitchen
4,2012-11-12 00:24:42,2012-11-12 00:24:54,Door,PIR,Living


## 3. Transform — Clean Sensor Events

- Convert strings to `datetime64`, drop rows that fail to parse
- Swap rows where `end_time < start_time` (guard)
- Build `sensor_id` as the `location_type_place` composite key, compute `duration_sec`
- Sort by start time, drop exact duplicates, assign `event_id`

In [3]:
sensors = sensors_raw.copy()

for col in ["start_time", "end_time"]:
    sensors[col] = pd.to_datetime(sensors[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")

n_invalid = int(sensors[["start_time", "end_time"]].isna().any(axis=1).sum())
sensors = sensors.dropna(subset=["start_time", "end_time"]).copy()

bad = sensors["end_time"] < sensors["start_time"]
sensors.loc[bad, ["start_time", "end_time"]] = sensors.loc[bad, ["end_time", "start_time"]].values

sensors["sensor_id"] = sensors["location"] + "_" + sensors["type"] + "_" + sensors["place"]
sensors["duration_sec"] = (sensors["end_time"] - sensors["start_time"]).dt.total_seconds().astype(int)

sensors = sensors.sort_values("start_time").drop_duplicates().reset_index(drop=True)
sensors.insert(0, "event_id", np.arange(1, len(sensors) + 1))

print(f"invalid rows dropped: {n_invalid}")
print(f"clean sensor events: {len(sensors)}")
sensors[['event_id', 'start_time', 'end_time', 'sensor_id', 'duration_sec']].head()

invalid rows dropped: 0
clean sensor events: 2334


,event_id,start_time,end_time,sensor_id,duration_sec
0,1,2012-11-11 21:14:21,2012-11-12 00:21:49,Seat_Pressure_Living,11248
1,2,2012-11-12 00:22:57,2012-11-12 00:22:59,Door_PIR_Living,2
2,3,2012-11-12 00:23:14,2012-11-12 00:23:17,Door_PIR_Kitchen,3
3,4,2012-11-12 00:24:20,2012-11-12 00:24:22,Door_PIR_Kitchen,2
4,5,2012-11-12 00:24:42,2012-11-12 00:24:54,Door_PIR_Living,12


## 4. Transform — Clean ADL Labels

Same rules apply: convert to datetime, strip whitespace, swap guard, sort & deduplicate, assign `adl_id`

In [4]:
adls = adls_raw.copy()

for col in ["start_time", "end_time"]:
    adls[col] = pd.to_datetime(adls[col], format="%Y-%m-%d %H:%M:%S", errors="coerce")
adls["activity"] = adls["activity"].str.strip()

adls = adls.dropna(subset=["start_time", "end_time", "activity"]).copy()

bad = adls["end_time"] < adls["start_time"]
adls.loc[bad, ["start_time", "end_time"]] = adls.loc[bad, ["end_time", "start_time"]].values
adls["duration_sec"] = (adls["end_time"] - adls["start_time"]).dt.total_seconds().astype(int)

adls = adls.sort_values("start_time").drop_duplicates().reset_index(drop=True)
adls.insert(0, "adl_id", np.arange(1, len(adls) + 1))

print(f"clean adl intervals: {len(adls)}")
adls.head()

clean adl intervals: 493


,adl_id,start_time,end_time,activity,duration_sec
0,1,2012-11-11 21:14:00,2012-11-12 00:22:59,Spare_Time/TV,11339
1,2,2012-11-12 00:24:00,2012-11-12 00:43:59,Spare_Time/TV,1199
2,3,2012-11-12 00:48:00,2012-11-12 00:49:59,Grooming,119
3,4,2012-11-12 00:50:00,2012-11-12 01:51:59,Spare_Time/TV,3719
4,5,2012-11-12 01:52:00,2012-11-12 01:52:59,Grooming,59


## 5. Transform — Join ADL Labels onto Events

- `merge_asof` (backward): match each event's start time to the ADL interval that began just before it
- If an event's end time extends past that ADL interval, mark it `Unlabeled` (covers label gaps)

In [5]:
adl_ref = (
    adls[["start_time", "end_time", "activity"]]
    .rename(columns={"end_time": "adl_end_time", "activity": "activity_label"})
    .sort_values("start_time")
)

events = pd.merge_asof(
    sensors.sort_values("start_time"),
    adl_ref,
    on="start_time",
    direction="backward",
)
events["activity_label"] = events["activity_label"].mask(
    events["end_time"] > events["adl_end_time"]
).fillna("Unlabeled")

print(events["activity_label"].value_counts(dropna=False).to_string())

activity_label
Unlabeled        915
Spare_Time/TV    464
Snack            170
Grooming         158
Breakfast        130
Lunch            126
Toileting        114
Leaving          103
Sleeping          78
Dinner            66
Showering         10


## 6. Quality (QA) Report

In [6]:
adl_sorted = adls.sort_values("start_time")
report = {
    "raw_sensor_rows": len(sensors_raw),
    "clean_sensor_rows": len(sensors),
    "dropped_invalid_rows": n_invalid,
    "unique_sensors": sensors["sensor_id"].nunique(),
    "raw_adl_rows": len(adls_raw),
    "clean_adl_rows": len(adls),
    "unique_activities": adls["activity"].nunique(),
    "overlapping_adl_intervals": int((adl_sorted["start_time"] < adl_sorted["end_time"].shift()).sum()),
    "date_range": [str(sensors["start_time"].min()), str(sensors["start_time"].max())],
    "unlabeled_events": int((events["activity_label"] == "Unlabeled").sum()),
}
report

{'raw_sensor_rows': 2334,
 'clean_sensor_rows': 2334,
 'dropped_invalid_rows': 0,
 'unique_sensors': 12,
 'raw_adl_rows': 493,
 'clean_adl_rows': 493,
 'unique_activities': 10,
 'overlapping_adl_intervals': 23,
 'date_range': ['2012-11-11 21:14:21', '2012-12-02 21:19:12'],
 'unlabeled_events': 915}

## 7. Build Dimension Tables

- `sensor_dim`: 12 composite sensor keys → location/type/place, event count, observed period
- `activity_dim`: 10 ADL activities → interval count, total duration (minutes)
- `events_fact`: event fact table (with ADL labels)

In [7]:
sensor_dim = (
    sensors.groupby("sensor_id", as_index=False)
    .agg(
        location=("location", "first"),
        type=("type", "first"),
        place=("place", "first"),
        event_count=("event_id", "count"),
        first_seen=("start_time", "min"),
        last_seen=("end_time", "max"),
    )
    .sort_values("event_count", ascending=False)
)
sensor_dim.insert(0, "sensor_key", np.arange(1, len(sensor_dim) + 1))

activity_dim = (
    adls.groupby("activity", as_index=False)
    .agg(
        interval_count=("adl_id", "count"),
        total_duration_sec=("duration_sec", "sum"),
    )
    .sort_values("interval_count", ascending=False)
)
activity_dim["total_duration_min"] = (activity_dim["total_duration_sec"] // 60).astype(int)
activity_dim = activity_dim.drop(columns="total_duration_sec")
activity_dim.insert(0, "activity_key", np.arange(1, len(activity_dim) + 1))

events_fact = events[["event_id", "start_time", "end_time", "sensor_id", "duration_sec", "activity_label"]].copy()
events_fact = events_fact.rename(columns={"activity_label": "activity"})

print(f"events_fact: {events_fact.shape}")
print(f"sensor_dim: {sensor_dim.shape}")
print(f"activity_dim: {activity_dim.shape}")
activity_dim.head()

events_fact: (2334, 6)
sensor_dim: (12, 8)
activity_dim: (10, 4)


,activity_key,activity,interval_count,total_duration_min
8,1,Spare_Time/TV,116,9037
2,2,Grooming,113,428
9,3,Toileting,93,169
7,4,Snack,47,407
3,5,Leaving,38,5269


## 8. Load — Save as CSV / Parquet

CSV uses `utf-8-sig` (Excel-compatible); Parquet is saved when `pyarrow` is installed

In [8]:
tables = {
    "events_fact": events_fact,
    "sensor_dim": sensor_dim,
    "activity_dim": activity_dim,
    "adl_intervals": adls,
}

for name, df in tables.items():
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

try:
    for name, df in tables.items():
        df.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
    parquet_saved = True
except ImportError:
    parquet_saved = False

print(f"CSV saved to {OUTPUT_DIR}")
print("Parquet: " + ("saved" if parquet_saved else "skipped (pyarrow not installed)"))

CSV saved to C:\Users\USER\Desktop\[16-08-26] Cloudy\07.Binary Dataset\processed
Parquet: saved


## 9. Verification — Re-check Saved Output

In [9]:
check = pd.read_csv(OUTPUT_DIR / "events_fact.csv", encoding="utf-8-sig")
print(f"reloaded events_fact: {check.shape}")
print(f"nulls: {int(check.isna().sum().sum())}")
print(f"sensor_id unique: {check['sensor_id'].nunique()}")
print(f"activity unique: {check['activity'].nunique()}")
check.head(3)

reloaded events_fact: (2334, 6)
nulls: 0
sensor_id unique: 12
activity unique: 11


,event_id,start_time,end_time,sensor_id,duration_sec,activity
0,1,2012-11-11 21:14:21,2012-11-12 00:21:49,Seat_Pressure_Living,11248,Spare_Time/TV
1,2,2012-11-12 00:22:57,2012-11-12 00:22:59,Door_PIR_Living,2,Spare_Time/TV
2,3,2012-11-12 00:23:14,2012-11-12 00:23:17,Door_PIR_Kitchen,3,Unlabeled
